<a href="https://colab.research.google.com/github/OutisAyo/council-classifier-/blob/main/notebooks/15_decision_support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning for Automated Classification and Prioritisation of Local Council Service Requests in the UK
## Notebook 15  Decision Support Layer

**Author:** Fashina Fuad Ayomide  
**MSc Data Science, University of South Wales**

This notebook builds the decision support layer combining rule-based response templates with retrieval-based historical context, plus a duplicate request detection system using text similarity.

## Mounting Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Importing the libraries

In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Loading the trained pipelines and dataset

In [3]:
processed_dir = '/content/drive/MyDrive/council-classifier/processed'
models_dir = '/content/drive/MyDrive/council-classifier/models'

department_pipeline = joblib.load(f'{models_dir}/department_pipeline_rf.pkl')
priority_pipeline = joblib.load(f'{models_dir}/priority_pipeline_final.pkl')

dataset = pd.read_csv(f'{processed_dir}/swansea_cleaned.csv')
dataset['date_received'] = pd.to_datetime(dataset['date_received'], errors='coerce')
dataset['date_closed'] = pd.to_datetime(dataset['date_closed'], errors='coerce')
dataset = dataset.dropna(subset=['priority', 'date_received'])

print("Dataset loaded:", dataset.shape)
print("Department pipeline loaded:", type(department_pipeline))
print("Priority pipeline loaded:", type(priority_pipeline))

Dataset loaded: (254377, 10)
Department pipeline loaded: <class 'sklearn.pipeline.Pipeline'>
Priority pipeline loaded: <class 'sklearn.pipeline.Pipeline'>


## Part 1: Rule-based response templates

We define a base template for each department, with wording that adjusts based on predicted priority.

In [4]:
RESPONSE_TEMPLATES = {
    'Waste Management': {
        'HIGH': "Thank you for reporting this waste management issue. Given the urgent nature of your request, it has been flagged for priority attention and assigned to our Waste Management team.",
        'MEDIUM': "Thank you for reporting this waste management issue. Your request has been logged and assigned to our Waste Management team for standard processing.",
        'LOW': "Thank you for reporting this waste management issue. Your request has been logged and will be addressed by our Waste Management team in due course."
    },
    'Pollution Division': {
        'HIGH': "Thank you for your report. Due to the urgent nature of this pollution-related issue, it has been escalated to our Pollution Division for immediate attention.",
        'MEDIUM': "Thank you for your report. This has been assigned to our Pollution Division for standard investigation.",
        'LOW': "Thank you for your report. This has been logged with our Pollution Division and will be reviewed in due course."
    },
    'Housing Division': {
        'HIGH': "Thank you for contacting us. Given the urgent nature of this housing matter, it has been escalated to our Housing Division for immediate review.",
        'MEDIUM': "Thank you for contacting us. This has been assigned to our Housing Division for standard review.",
        'LOW': "Thank you for contacting us. This has been logged with our Housing Division and will be reviewed in due course."
    },
    'Licensing Division': {
        'HIGH': "Thank you for your enquiry. This licensing matter has been flagged as urgent and escalated to our Licensing Division.",
        'MEDIUM': "Thank you for your enquiry. This has been assigned to our Licensing Division for standard processing.",
        'LOW': "Thank you for your enquiry. This has been logged with our Licensing Division and will be processed in due course."
    },
    'Food and Safety Division': {
        'HIGH': "Thank you for your report. Given the urgent food safety concern raised, this has been escalated to our Food and Safety Division for immediate investigation.",
        'MEDIUM': "Thank you for your report. This has been assigned to our Food and Safety Division for standard investigation.",
        'LOW': "Thank you for your report. This has been logged with our Food and Safety Division and will be reviewed in due course."
    },
    'Trading Standards Division': {
        'HIGH': "Thank you for your report. This has been flagged as urgent and escalated to our Trading Standards Division for immediate review.",
        'MEDIUM': "Thank you for your report. This has been assigned to our Trading Standards Division for standard review.",
        'LOW': "Thank you for your report. This has been logged with our Trading Standards Division and will be reviewed in due course."
    }
}

print("Templates defined for", len(RESPONSE_TEMPLATES), "departments")

Templates defined for 6 departments


## Part 2: Retrieval-based historical context

For a new request, we find the most similar historical requests using cosine similarity on TF-IDF vectors, then report their typical resolution time. This grounds the template response in real historical patterns rather than generic wording alone.

In [5]:
retrieval_vectorizer = TfidfVectorizer(max_features=1500)
historical_vectors = retrieval_vectorizer.fit_transform(dataset['request_text'])

print("Historical vector matrix shape:", historical_vectors.shape)

Historical vector matrix shape: (254377, 591)


In [6]:
def find_similar_requests(new_text, top_n=5):
    """
    Find the most similar historical requests to a new request text,
    and return their average resolution time and most common priority.
    """
    new_vector = retrieval_vectorizer.transform([new_text])
    similarities = cosine_similarity(new_vector, historical_vectors).flatten()

    top_indices = similarities.argsort()[-top_n:][::-1]
    similar_requests = dataset.iloc[top_indices].copy()
    similar_requests['similarity_score'] = similarities[top_indices]

    valid_resolution = similar_requests[
        (similar_requests['date_closed'].notna()) &
        ((similar_requests['date_closed'] - similar_requests['date_received']).dt.days >= 0)
    ]

    if len(valid_resolution) > 0:
        avg_days = (valid_resolution['date_closed'] - valid_resolution['date_received']).dt.days.mean()
    else:
        avg_days = None

    return similar_requests[['request_text', 'assigned_department', 'priority', 'similarity_score']], avg_days

# Quick test
test_text = "live rat in house urgent"
similar, avg_resolution = find_similar_requests(test_text)
print("Most similar historical requests:")
print(similar)
print(f"\nAverage historical resolution time for similar requests: {avg_resolution:.1f} days" if avg_resolution else "No resolution data available")

Most similar historical requests:
                       request_text assigned_department priority  \
123967  Live Rat in House * Urgent*  Pollution Division      LOW   
240570  Live Rat in House * Urgent*  Pollution Division   MEDIUM   
10924   Live Rat in House * Urgent*  Pollution Division     HIGH   
153359  Live Rat in House * Urgent*  Pollution Division     HIGH   
159007  Live Rat in House * Urgent*  Pollution Division     HIGH   

        similarity_score  
123967               1.0  
240570               1.0  
10924                1.0  
153359               1.0  
159007               1.0  

Average historical resolution time for similar requests: 23.8 days


## Part 3: Combining templates with retrieval into one response generator

In [7]:
def generate_response(request_text, day_of_week, month):
    """
    Full decision support pipeline: predicts department and priority,
    retrieves similar historical context, and generates a combined response.
    """
    predicted_department = department_pipeline.predict([request_text])[0]

    priority_input = pd.DataFrame({
        'request_text': [request_text],
        'day_of_week': [day_of_week],
        'month': [month]
    })
    predicted_priority = priority_pipeline.predict(priority_input)[0]

    template = RESPONSE_TEMPLATES.get(predicted_department, {}).get(
        predicted_priority,
        "Thank you for your request. It has been logged and will be reviewed."
    )

    similar_requests, avg_resolution = find_similar_requests(request_text)

    if avg_resolution is not None:
        context_line = f" Based on similar requests, this typically takes around {avg_resolution:.0f} days to resolve."
    else:
        context_line = ""

    full_response = template + context_line

    return {
        'predicted_department': predicted_department,
        'predicted_priority': predicted_priority,
        'response': full_response,
        'similar_requests': similar_requests
    }

# Test the full pipeline
result = generate_response("live rat in house urgent", day_of_week=1, month=6)
print("Predicted Department:", result['predicted_department'])
print("Predicted Priority:", result['predicted_priority'])
print("\nGenerated Response:")
print(result['response'])

Predicted Department: Pollution Division
Predicted Priority: HIGH

Generated Response:
Thank you for your report. Due to the urgent nature of this pollution-related issue, it has been escalated to our Pollution Division for immediate attention. Based on similar requests, this typically takes around 24 days to resolve.


## Part 4: Duplicate request detection

We check whether a new request is highly similar to other requests submitted within a recent time window a proxy for detecting when multiple citizens report the same underlying issue.

In [8]:
def check_for_duplicates(new_text, recent_days=7, similarity_threshold=0.7, top_n=5):
    """
    Checks whether a new request closely matches any request received
    within the last `recent_days`, based on cosine similarity.
    """
    most_recent_date = dataset['date_received'].max()
    cutoff_date = most_recent_date - pd.Timedelta(days=recent_days)
    recent_requests = dataset[dataset['date_received'] >= cutoff_date].copy()

    if len(recent_requests) == 0:
        return pd.DataFrame(), False

    recent_vectors = retrieval_vectorizer.transform(recent_requests['request_text'])
    new_vector = retrieval_vectorizer.transform([new_text])

    similarities = cosine_similarity(new_vector, recent_vectors).flatten()
    recent_requests['similarity_score'] = similarities

    duplicates = recent_requests[recent_requests['similarity_score'] >= similarity_threshold]
    duplicates = duplicates.sort_values('similarity_score', ascending=False).head(top_n)

    is_likely_duplicate = len(duplicates) > 0

    return duplicates[['request_text', 'assigned_department', 'date_received', 'similarity_score']], is_likely_duplicate

# Test duplicate detection
duplicates_found, is_duplicate = check_for_duplicates("recycling bag not collected")
print("Is this likely a duplicate?", is_duplicate)
print("\nMatching recent requests:")
print(duplicates_found)

Is this likely a duplicate? False

Matching recent requests:
Empty DataFrame
Columns: [request_text, assigned_department, date_received, similarity_score]
Index: []


## Testing the complete decision support layer with multiple examples

In [9]:
test_cases = [
    ("live rat in house urgent", 1, 6),
    ("recycling bag not collected", 4, 3),
    ("HMO licence application", 2, 9),
]

for text, dow, month in test_cases:
    print("="*70)
    print(f"Request: '{text}'")
    result = generate_response(text, dow, month)
    print(f"Department: {result['predicted_department']}")
    print(f"Priority: {result['predicted_priority']}")
    print(f"Response: {result['response']}")
    duplicates, is_dup = check_for_duplicates(text)
    print(f"Potential duplicate flagged: {is_dup}")
    print()

Request: 'live rat in house urgent'
Department: Pollution Division
Priority: HIGH
Response: Thank you for your report. Due to the urgent nature of this pollution-related issue, it has been escalated to our Pollution Division for immediate attention. Based on similar requests, this typically takes around 24 days to resolve.
Potential duplicate flagged: False

Request: 'recycling bag not collected'
Department: Waste Management
Priority: LOW
Response: Thank you for reporting this waste management issue. Your request has been logged and will be addressed by our Waste Management team in due course. Based on similar requests, this typically takes around 796 days to resolve.
Potential duplicate flagged: False

Request: 'HMO licence application'
Department: Housing Division
Priority: LOW
Response: Thank you for contacting us. This has been logged with our Housing Division and will be reviewed in due course. Based on similar requests, this typically takes around 116 days to resolve.
Potential d

## Saving the decision support components

We save the retrieval vectorizer, the historical vectors, and the templates dictionary so the dashboard can load and use them directly without recomputation.

In [10]:
import pickle

decision_support_dir = '/content/drive/MyDrive/council-classifier/models'

joblib.dump(retrieval_vectorizer, f'{decision_support_dir}/retrieval_vectorizer.pkl')

with open(f'{decision_support_dir}/response_templates.pkl', 'wb') as f:
    pickle.dump(RESPONSE_TEMPLATES, f)

dataset.to_pickle(f'{decision_support_dir}/dataset_for_retrieval.pkl')

print("Decision support components saved to:", decision_support_dir)

Decision support components saved to: /content/drive/MyDrive/council-classifier/models
